# L6: Colaboração Multiagente para Análise Financeira
* Nesta lição, você aprenderá maneiras de fazer com que os agentes colaborem entre si.

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from utils import get_openai_api_key, pretty_print_result
from utils import get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

# Creating Agents

In [5]:
data_analyst_agent = Agent(
    role="Analista de Dados",
    goal="Monitorar e analisar dados de mercado em tempo real "
         "para identificar tendências e prever movimentos de mercado.",
    backstory="Especializado em mercados financeiros, este agente "
              "utiliza modelagem estatística e aprendizado de máquina "
              "para fornecer insights cruciais. Com habilidade para lidar com dados, "
              "o Agente Analista de Dados é fundamental para "
              "informar decisões de negociação.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool]
)

In [6]:
trading_strategy_agent = Agent(
    role="Desenvolvedor de Estratégias de Negociação",
    goal="Desenvolver e testar diversas estratégias de negociação com base "
         "em insights do Agente Analista de Dados.",
    backstory="Com um profundo entendimento dos mercados financeiros "
              "e análise quantitativa, este agente cria e refina estratégias "
              "de negociação. Ele avalia o desempenho de diferentes abordagens "
              "para determinar as opções mais lucrativas e avessas ao risco.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool]
)

In [7]:
execution_agent = Agent(
    role="Consultor de Negociação",
    goal="Sugerir estratégias ideais de execução de negociações "
         "com base em estratégias de negociação aprovadas.",
    backstory="Este agente é especializado em analisar o momento, preço "
              "e detalhes logísticos de negociações potenciais. Avaliando "
              "esses fatores, ele fornece sugestões bem fundamentadas sobre "
              "quando e como as negociações devem ser executadas para maximizar "
              "a eficiência e a aderência à estratégia.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool]
)

In [8]:
risk_management_agent = Agent(
    role="Consultor de Riscos",
    goal="Avaliar e fornecer insights sobre os riscos "
         "associados a atividades de negociação potenciais.",
    backstory="Com um profundo entendimento de modelos de avaliação de riscos "
              "e dinâmicas de mercado, este agente analisa minuciosamente os riscos "
              "potenciais das negociações propostas. Ele oferece uma análise detalhada "
              "da exposição ao risco e sugere salvaguardas para garantir que as atividades "
              "de negociação estejam alinhadas com a tolerância ao risco da empresa.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool]
)

# Creating Tasks

In [9]:
# Tarefa para o Agente Analista de Dados: Analisar Dados de Mercado
data_analysis_task = Task(
    description=(
        "Monitore e analise continuamente os dados de mercado para "
        "a ação selecionada ({stock_selection}). "
        "Use modelagem estatística e aprendizado de máquina para "
        "identificar tendências e prever movimentos do mercado."
    ),
    expected_output=(
        "Insights e alertas sobre oportunidades ou ameaças significativas "
        "no mercado para {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [10]:
# Tarefa para o Agente de Estratégias de Negociação: Desenvolver Estratégias de Negociação
strategy_development_task = Task(
    description=(
        "Desenvolva e refine estratégias de negociação com base nos "
        "insights do Analista de Dados e na tolerância ao risco definida pelo usuário ({risk_tolerance}). "
        "Considere as preferências de negociação ({trading_strategy_preference})."
    ),
    expected_output=(
        "Um conjunto de estratégias de negociação potenciais para {stock_selection} "
        "que estejam alinhadas com a tolerância ao risco do usuário."
    ),
    agent=trading_strategy_agent,
)

In [11]:
# Tarefa para o Agente Consultor de Negociação: Planejar Execução de Negociações
execution_planning_task = Task(
    description=(
        "Analise as estratégias de negociação aprovadas para determinar os "
        "melhores métodos de execução para {stock_selection}, "
        "considerando as condições atuais do mercado e o preço ideal."
    ),
    expected_output=(
        "Planos de execução detalhados sugerindo como e quando "
        "executar negociações para {stock_selection}."
    ),
    agent=execution_agent,
)

In [12]:
# Tarefa para o Agente Consultor de Riscos: Avaliar Riscos de Negociação
risk_assessment_task = Task(
    description=(
        "Avalie os riscos associados às estratégias de negociação propostas "
        "e aos planos de execução para {stock_selection}. "
        "Forneça uma análise detalhada dos riscos potenciais "
        "e sugira estratégias de mitigação."
    ),
    expected_output=(
        "Um relatório abrangente de análise de riscos detalhando os riscos "
        "potenciais e as recomendações de mitigação para {stock_selection}."
    ),
    agent=risk_management_agent,
)

# Creating the Crew

* A classe Process ajuda a delegar o fluxo de trabalho aos Agents (como um Manager no trabalho)
* No exemplo abaixo, ele executará isso hierarquicamente.
* manager_llm permite que você escolha o LLM "manager" que deseja usar.

In [13]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define a equipe com agentes e tarefas
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

# Running the Crew

In [14]:
# Exemplo de dados para iniciar o processo
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Médio',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

In [15]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Monitore e analise continuamente os dados de mercado para a ação selecionada (AAPL). Use modelagem estatística e aprendizado de máquina para identificar tendências e prever movimentos do mercado.


> Entering new CrewAgentExecutor chain...
I need to ensure that the team is effectively monitoring and analyzing market data for the selected stock (AAPL) in order to identify trends and predict market movements using statistical modeling and machine learning techniques.

Action: Delegate work to co-worker
Action Input: {"coworker": "Analista de Dados", "task": "Monitore e analise continuamente os dados de mercado para a ação selecionada (AAPL) utilizando modelagem estatística e aprendizado de máquina", "context": "É fundamental acompanhar diariamente os dados de mercado da ação AAPL, aplicar técnicas de modelagem estatística e aprendizado de máquina para identificar tendências e prever movimentos no mercado."}

> Entering new Cre

In [16]:
from IPython.display import Markdown
Markdown(result)

Desculpe, não consegui acessar o conteúdo do site da AAPL devido a restrições de segurança. Recomendo que entremos em contato com o proprietário do site para obter as informações necessárias sobre as estratégias de negociação e planos de execução da empresa.